# EXP-2026-008 / Q5-E - Leg 2 failure mechanism audit

Implementation only. This notebook is committed **unexecuted**: every output
cell is empty and no cell has an execution count.

Running the audit on the registered artifacts needs a **separate user
approval that does not exist yet**. Two independent switches keep it closed:
`OPEN_REGISTERED_DATA` defaults to `False`, and the production route also
requires an explicit approval token. Either one alone refuses.

Read the frozen design first:
`experiments/specs/EXP-2026-008-q5e-leg2-failure-mechanism-audit.md`.

Order of the cells below is the order of the contract: setup, staleness
guard, constants, switches, per-stage announcement, production route.

In [ ]:
# 1. Setup. Declared dependencies are checked before anything is read.
REPO = '/content/repo'
import os, sys
if not os.path.exists(REPO):
    REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(REPO, 'mit-bih'))

import q5d_order_preserving_beat_join as BJ
import q5e_leg2_failure_mechanism_audit as Q5E

print('Q5-E module :', Q5E.__file__)
print('frozen Q5-D :', BJ.__file__)

In [ ]:
# 2. Staleness guard. A version integer is defeated by forgetting to bump it,
#    so assert the capabilities actually used - on BOTH modules.
MISSING_Q5E = [n for n in Q5E.module_capabilities() if not hasattr(Q5E, n)]
NEED_BJ = ('candidate_edges', 'match_record', 'rule_fingerprint',
           'cache_expected_files', 'hash_file_set', 'RecordSequence')
MISSING_BJ = [n for n in NEED_BJ if not hasattr(BJ, n)]
assert not MISSING_Q5E, f'stale Q5-E clone, missing {MISSING_Q5E}'
assert not MISSING_BJ, f'stale Q5-D clone, missing {MISSING_BJ}'
assert BJ.rule_fingerprint() == Q5E.REGISTERED_RULE_FINGERPRINT, \
    'frozen Q5-D rule fingerprint moved'
print('capabilities OK; frozen fingerprint', BJ.rule_fingerprint())

In [ ]:
# 3. Constants card. This opens nothing and is not a result.
print(Q5E.design_card())
print()
print(Q5E.assert_implementation_only())

In [ ]:
# 4. Switches. Both default to closed and stay that way until the separate
#    execution approval exists. Do not edit these to "try it out".
MODE = Q5E.MODE_DESIGN
APPROVAL = None
OPEN_REGISTERED_DATA = False

print('MODE                :', MODE)
print('approval present    :', Q5E.execution_is_approved(APPROVAL))
print('OPEN_REGISTERED_DATA:', OPEN_REGISTERED_DATA)
print()
print(Q5E.APPROVAL_NOTE)

In [ ]:
# 5. Runtime dependency table for the selected stage, before any work.
REPORT = Q5E.check_runtime_dependencies(MODE)
for name, (why, pinned) in sorted(Q5E.RUNTIME_DEPENDENCIES.items()):
    print(f'{name:12s} {pinned or "-":8s} {why}')
print()
print('required for this stage:', REPORT['required'])
print('missing                :', REPORT['missing'])
if REPORT['missing']:
    print('install first          :', REPORT['pip_install'])

In [ ]:
# 6. Every stage announces RUN or SKIP with its reason. A stage that quietly
#    does nothing must never look like a stage that passed.
STAGES = ('QA', 'M0', 'M1', 'M2', 'M3', 'M4-gate', 'controls', 'decision')
WILL_RUN = {s: Q5E.stage_should_run(s, MODE, APPROVAL) for s in STAGES}
print()
print('stages that would run:', [s for s, v in WILL_RUN.items() if v])

In [ ]:
# 7. Figure contract. Titles and axes are ASCII so Colab cannot render a
#    missing glyph into a plot that is then cited as a result. Each figure
#    declares its own kind and panels: two figures sharing a series is a
#    rendering bug, and render_figures refuses it rather than writing twice.
SPECS = Q5E.figure_specs(m4_ok=True)
Q5E.assert_ascii_labels(SPECS)
assert len({s['kind'] for s in SPECS}) == len(SPECS), 'figure kinds collide'
for spec in SPECS:
    print(f"{spec['file']:44s} {spec['title']}")
    print(f"{'':44s} panels: {', '.join(spec['panels'])}")
print()
print('figure 7 is produced only when the M4 gate passes; its absence is')
print('recorded in the bundle together with the reason.')

In [ ]:
# 8. M3 candidate-graph panels. BOTH sides are shown, and each panel states
#    its decisional status so a reader cannot mistake one for the other.
#    Codex fixed the H4 side to 'cache' on 2026-08-12, before any execution.
SIDE_LABEL = {
    Q5E.SIDE_CACHE: 'H4 decisional',
    Q5E.SIDE_MAMBA: 'descriptive, non-decisional',
}
print('H4 decisional side:', Q5E.H4_DECISIONAL_SIDE)
print()
for side in Q5E.SIDES:
    print(f'M3 panel: {side:6s} [{SIDE_LABEL[side]}]')
    for group in Q5E.M3_GROUPS:
        print(f'    {group:12s} degree / rr_pair_multiplicity / local_rr_sd')
print()
print('The mamba panel never enters an H4 p-value, a q99 comparison, an\n'
      'effect gate, the Holm adjustment, an association flag, or the\n'
      'decision tree. It is kept so the candidate graph stays symmetric.')

In [ ]:
# 9. Input resolution. No Drive path is typed by hand, and no stamp is
#    trusted. Discovery finds each input by digest; run_audit then re-verifies
#    ALL of them from their bytes immediately before running, so a mapping
#    assembled by hand is refused however plausible it looks. Byte-identical
#    duplicates resolve deterministically and are recorded -- Drive holds
#    duplicates, and no Drive file is deleted or moved to satisfy this.
#
#    The bundle is checked as TWO separate contracts, because they answer
#    different questions and conflating them rejected real bundles:
#      directory contract : the whole frozen 12-file Q5-D run bundle
#      input identity     : the 5 files Q5-E reads, folded as a subset so the
#                           other 7 registered files are not "unexpected"
SEARCH_ROOT = '/content/drive/MyDrive'   # the mount root only, nothing deeper
OUT_DIR = '/content/drive/MyDrive/medkos_runs'

print('search root       :', SEARCH_ROOT)
print('resolves by       : registered digest, then re-verified from bytes')
print('bundle directory  :', len(BJ.BUNDLE_FILES), 'registered files')
print('Q5-E reads        :', len(Q5E.BUNDLE_INPUT_FILES), 'of them')
print('V10 source        :', ', '.join(Q5E.M4_V10_SOURCE_FILES))
print('MIT-BIH           : publisher', BJ.MITDB_CHECKSUM_FILE, '+ aggregate')
print('refuses on        : zero matches, differing digests, SUPERSEDED')
print()
print('THREE registration items are open. Each is a terminal stop, not a')
print('warning: an approved run halts on them rather than proceeding on a')
print('weaker check. Nothing here guesses or fills in a value.')
print('  P1 MIT-BIH tree aggregate :',
      Q5E.MITDB_TREE_AGGREGATE or Q5E.INPUT_IDENTITY_REGISTRATION_REQUIRED)
print('  P2 canonical bundle sha   :',
      Q5E.SOURCE_BUNDLE_FILE_SHA256 or Q5E.SOURCE_BUNDLE_DIGEST_FREEZE_REQUIRED)
print('  P3 source-match adapter   :',
      Q5E.source_match_equivalence_status()['status'])
print()
print('P3 is an M4.0 sub-gate that runs BEFORE the detector:')
print('  gate order    :', Q5E.M4_GATE_ORDER)
print('  before replay :', Q5E.M4_GATES_BEFORE_REPLAY)
print('The annotation-matching adapter is a text-derived candidate and is')
print('unverified against the registered data.py, so the detector must not')
print('run through it yet. A PASS needs the full differential record:')
print('  required fixtures :', len(Q5E.SOURCE_MATCH_REQUIRED_FIXTURES))
print('  digest fields     :', ', '.join(Q5E.SOURCE_MATCH_ORACLE_DIGEST_FIELDS))
print('Designs: research/HANDOFF_2026-08-12_Q5E_prep_p1p2p3_to_codex.md.')
print('Each PREP needs its own user approval.')
print()
print('Only the five registered Leg 2 inputs are resolved here. Every sealed')
print('artifact named in the spec stays sealed and is never located, opened')
print('or hashed by this notebook.')

In [ ]:
# 10. Production route. This is the ONLY path to a Q5-E decision, and it is
#     complete: M4.0 re-runs the registered detector on all 22 DS1 records
#     through the digest-verified producer, M4.1 places anchors from that same
#     matching, M5 stratifies every hypothesis statistic, and the bundle is
#     staged and verified before it is published. When the separate execution
#     approval exists, setting the two switches in cell 4 is the whole change.
#     With the switches as committed it refuses, which is the intended state.
if Q5E.stage_should_run('run_audit', MODE, APPROVAL):
    RESULT = Q5E.run_audit_from_mount(
        SEARCH_ROOT, OUT_DIR,
        approval=APPROVAL,
        open_registered_data=OPEN_REGISTERED_DATA)
    print('decision            :', RESULT['decision'])
    print('QA target set       :', RESULT['qa_target_set'])
    print('M4 status           :', RESULT['m4_status'])
    # The real first failing sub-gate, not just the umbrella status: this is
    # how SOURCE_MATCH_EQUIVALENCE_REQUIRED is told apart from a count or RR
    # mismatch when M4 stops.
    print('first stop reason   :', RESULT['first_stopping_reason'])
    # Whether the detector actually ran is a different fact from whether M4
    # passed -- a replay can run and then fail the count or RR sub-gate.
    print('detector ran        :', RESULT['detector_replay_performed'])
    for name in ('H1', 'H2', 'H3', 'H4'):
        entry = RESULT['tests'][name]
        print(f"  {name}: p_holm_4family={entry['p_holm_4family']} "
              f"flag={entry.get('flag')} "
              f"strata={entry.get('strata_reported')}")
    print()
    print('A flag needs a real non-pooled stratified statistic, not merely a')
    print('stratum name. The bundle is staged, verified and only then')
    print('published, so the output path never holds a partial run.')
    print('If M4 stops, M0-M3 remain as diagnostic partial results and the')
    print('registered decision branch is still reached.')
else:
    print('run_audit not started. Nothing above this line is a measurement.')

## What this notebook may not do

No registered aggregation, no detector reproduction, no beat-join re-run, no
per-beat label of the held-out split, no model score, no association, no
training, and no write to an existing bundle. The audit reports **associated
mechanisms** only; it never states that an association is a cause, and it
licenses no change to the frozen Q5-D join rule.

Next steps, in order: this implementation PR is reviewed, the user separately
approves execution, and only then may the stages above run and write one new
timestamped bundle.